# Day 3: Context Managers

## Context Managers in Python

A **context manager** allows you to manage resources automatically. 
It handles setup and cleanup operations so you don't have to do it manually.

Common use cases:
- Opening and closing files
- Acquiring and releasing locks
- Database connections
- Temporary state changes



In [6]:
# Example: Reading a file using a context manager

with open('file.txt', 'w') as f:
    f.write("Hello, Context Managers!\n")

with open('file.txt', 'r') as f:
    content = f.read()
    print(content)

# The file is automatically closed after the 'with' block


Hello, Context Managers!



>> Explanation: The with statement ensures that the file is closed automatically, even if an error occurs inside the block.

## The Context Manager Protocol




In [3]:
class MyContext:
    def __enter__(self):
        print("1. Entering the context")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print("3. Exiting the context")

with MyContext() as ctx:
    print("2. Inside the context")

1. Entering the context
2. Inside the context
3. Exiting the context


In [7]:
class MyContextManager:
    def __enter__(self):
        print("Entering context")
        return self  # This object is assigned to 'cm'

    def __exit__(self, exc_type, exc_val, exc_tb):
        print("Exiting context")
        # Returning False propagates exceptions, True suppresses them
        return False

with MyContextManager() as cm:
    print("Inside context")
    # Uncomment the next line to see how exception handling works
    1 / 0

Entering context
Inside context
Exiting context


ZeroDivisionError: division by zero

## Creating Custom Context Managers


### Example 1: File Handler with Logging

In [8]:
class FileHandler:
    def __init__(self, filename, mode='r'):
        self.filename = filename
        self.mode = mode
        self.file = None

    def __enter__(self):
        print(f"Opening {self.filename}")
        self.file = open(self.filename, self.mode)
        return self.file

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.file:
            print(f"Closing {self.filename}")
            self.file.close()
        return False

with FileHandler("file.txt", 'r') as f:
    content = f.read()
    print(content)

Opening file.txt
Hello, Context Managers!

Closing file.txt


### Example 2: Database Connection Manager (Simulated)

In [9]:
class DatabaseConnection:
    def __init__(self, db_name):
        self.db_name = db_name
        self.connection = None

    def __enter__(self):
        print(f"Connecting to {self.db_name}")
        # Simulate a database connection
        self.connection = f"Connection to {self.db_name}"
        return self.connection

    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f"Closing connection to {self.db_name}")
        self.connection = None
        return False

with DatabaseConnection('mydb') as conn:
    print(f"Using {conn}")

Connecting to mydb
Using Connection to mydb
Closing connection to mydb


## Exception Handling in Context Managers

Context managers can intercept and handle exceptions that occur inside a `with` block.
This is done using the `__exit__` method, which receives information about the exception.

A context manager can:
- Detect whether an exception occurred
- Access exception details
- Decide whether to suppress or propagate the exception

### Understanding __exit__ Parameters

#### The `__exit__` Method Parameters

The `__exit__` method receives three arguments related to exceptions:

- `exc_type`: The exception class (e.g., `ValueError`)
- `exc_val`: The exception instance (error message)
- `exc_tb`: The traceback object

If no exception occurs, all three values are `None`.


### Suppressing an Exception

In [10]:
class SafeOperation:
    def __enter__(self):
        print("Entering the context")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None:
            print(f"Exception occured : {exc_type.__name__} : {exc_val}")
            return True # Suppress the exception
        return False

#### Using the Context Manager (Exception Suppressed)

In [11]:
with SafeOperation():
    print("Inside the context")
    raise ValueError("Something went wrong")

print("Execution continues after the with block")

Entering the context
Inside the context
Exception occured : ValueError : Something went wrong
Execution continues after the with block


#### Expected behavior:
- The exception is caught inside __exit__
- No traceback is shown
- Program execution continues normally

### Propagating an Exception

In [12]:
class UnsafeOperation:
    def __enter__(self):
        print("Entering the context")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None:
            print(f"Exception detected: {exc_type.__name__}: {exc_val}")
        return False

#### Using the Context Manager (Exception Propagated)

In [13]:
with UnsafeOperation():
    print("Inside the context")
    raise RuntimeError("This error will propagate")

print("This line will not execute")

Entering the context
Inside the context
Exception detected: RuntimeError: This error will propagate


RuntimeError: This error will propagate

#### Expected behavior:
- __exit__ is still executed
- The exception is re-raised
- Jupyter displays the full traceback

#### One-Line Mental Model

> **Returning `True` = “I handled it, move on.”**  
> **Returning `False` = “I didn’t handle it, crash if needed.”**


#### Key Observations


- `__exit__` is always called, even if an exception occurs
- Returning `True` suppresses the exception
- Returning `False` allows the exception to propagate
- Jupyter Notebook displays full tracebacks for propagated exceptions
- Context managers allow controlled, centralized exception handling


## The contextlib Module


### Understanding `yield` in `contextlib` Context Managers

When using `contextlib.contextmanager`, the `yield` keyword is used to split a function
into two phases:

1. Setup phase (before `yield`)
2. Cleanup phase (after `yield`)

The code before `yield` behaves like `__enter__`,
and the code after `yield` behaves like `__exit__`.


### Basic Flow of `yield`

In [14]:
from contextlib import contextmanager

@contextmanager
def simple_context():
    print("Before yield (entering context)")
    yield
    print("After yield (exiting context)")

with simple_context():
    print("Inside with block")

Before yield (entering context)
Inside with block
After yield (exiting context)


### `yield` as a Value Provider

The value passed to `yield` is assigned to the variable after the `as` keyword.


In [15]:
@contextmanager
def resource_provider():
    resource = "Important Resource"
    yield resource

with resource_provider() as res:
    print(res)

Important Resource


### Why `yield` Must Appear Exactly Once

In a `@contextmanager` function:
- One `yield` represents entering the context
- Resuming after `yield` represents exiting the context

Multiple `yield` statements would break this entry/exit contract.


### `yield` and Exception Safety

If an exception occurs inside the `with` block:
- Execution resumes after `yield`
- Cleanup code still runs
- The exception is re-raised unless explicitly handled


In [16]:
@contextmanager
def safe_context():
    print("setup")
    try:
        yield
    finally:
        print("Cleanup")

with safe_context():
    print("Inside context")
    raise ValueError("Error inside context")

setup
Inside context
Cleanup


ValueError: Error inside context

### Timer Context Manager

In [17]:
import time
from contextlib import contextmanager

@contextmanager
def timer(name="Operation"):
    start = time.time()
    print(f"Starting {name}")
    try:
        yield
    finally:
        end = time.time()
        print(f"{name} took {end-start:.4f} seconds")

with timer("Data processing"):
    time.sleep(1)
    # perform some operation

Starting Data processing
Data processing took 1.0051 seconds


##### Explanation:

- Timing starts before entering the block
- Timing ends after the block exits
- Execution time is printed regardless of success or failure

### Example: Temporary Directory Context Manager


In [19]:
import os
import tempfile
import shutil
from contextlib import contextmanager

@contextmanager
def temporary_directory():
    temp_dir = tempfile.mkdtemp()
    try:
        yield temp_dir
    finally:
        shutil.rmtree(temp_dir)

with temporary_directory() as tmpdir:
    filepath = os.path.join(tmpdir, 'temp.txt')
    with open(filepath, 'w') as f:
        f.write("Temporary data")


This logic creates a temporary folder for you, lets you use it for some work, and then automatically deletes it when you are done.

First, a new temporary directory is created on the system. This directory exists only for a short time and is meant for temporary files.

Then, control is given to you so you can use that directory. While you are using it, you can create files inside it and work with them normally.

Once your work is finished and you exit the block, Python automatically cleans up by deleting the temporary directory along with everything inside it. This cleanup happens even if something goes wrong during your work.

#### Mapping `yield` to the Context Manager Protocol

| `@contextmanager` Code | Protocol Equivalent |
|-----------------------|---------------------|
| Code before `yield`   | `__enter__`         |
| Value yielded         | Return value of `__enter__` |
| Code after `yield`    | `__exit__`          |


#### Key Takeaways

- `yield` pauses execution and hands control to the `with` block
- Code before `yield` runs when entering the context
- Code after `yield` runs when exiting the context
- Cleanup code should be placed in a `finally` block
- `contextlib` replaces boilerplate `__enter__` and `__exit__`


## Useful contextlib Utilities


### Useful `contextlib` Utilities

The `contextlib` module provides helper context managers that simplify
exception handling and resource cleanup.

In this section, we explore:
- `suppress()` for ignoring specific exceptions
- `closing()` for ensuring cleanup
- `ExitStack` for managing multiple context managers dynamically



### `suppress()`

`suppress()` is used to ignore specific exceptions that are expected
and non-critical. If the given exception occurs, it is silently ignored.


In [20]:
from contextlib import suppress
import os

with suppress(FileNotFoundError):
    os.remove("nonexistent_file.txt")

print("Program continues normally")

Program continues normally


**Observation:**

-   No error is raised if the file does not exist
    
-   Other exceptions are not suppressed
    

#### `closing()`

`closing()` ensures that an object's `close()` method is called
when exiting the `with` block.

This is useful for objects that do not implement the context manager protocol.


In [21]:
from contextlib import closing
from urllib.request import urlopen

with closing(urlopen("http://example.com")) as page:
    content = page.read()
    print("Read", len(content), "bytes")

Read 513 bytes


**Observation:**

-   `page.close()` is automatically called
    
-   Prevents resource leaks


#### `ExitStack`

`ExitStack` allows you to manage multiple context managers dynamically.
This is useful when the number of resources is not known ahead of time.

In [22]:
from contextlib import ExitStack

file_list = ["file1.txt", "file2.txt"]

# Create example files
for name in file_list:
    with open(name, 'w') as f:
        f.write(f"Contents of {name}")

with ExitStack() as stack:
    files = [stack.enter_context(open(name)) for name in file_list]
    for f in files:
        print(f.read())

Contents of file1.txt
Contents of file2.txt


**Observation:**

-   All files are opened safely
    
-   All files are closed automatically on exit
    
-   Cleanup happens in reverse order


#### When to Use Which Utility

- Use `suppress()` to ignore expected errors
- Use `closing()` for objects with a `close()` method
- Use `ExitStack` when managing many or conditional resources

These utilities reduce boilerplate and make resource handling safer.


#### Key Takeaways

- `contextlib` provides ready-to-use context managers
- These utilities simplify exception handling and cleanup
- `ExitStack` is powerful for dynamic resource management
- Cleaner code leads to fewer bugs and leaks


## Common Use Cases of Context Managers (Runnable Examples)

This section demonstrates practical, real-world use cases of context managers:
- Lock management
- Temporary directory change
- Temporary environment variables

Each example is runnable and shows automatic cleanup.


### Example 1. Lock Management (Thread Safety)

**What you’ll observe**

- Lock is acquired before entering the block

- Lock is released automatically after the block

In [23]:
import threading
from contextlib import contextmanager
import time

@contextmanager
def acquired_lock(lock):
    lock.acquire()
    try:
        print("Lock acquired")
        yield
    finally:
        lock.release()
        print("Lock released")

lock = threading.Lock()

with acquired_lock(lock):
    print("Inside critical section")
    time.sleep(1)


Lock acquired
Inside critical section
Lock released


### Example 2: Temporarily Changing the Working Directory

This context manager changes the current directory only inside the `with` block
and restores it afterward.


In [24]:
import os
from contextlib import contextmanager

@contextmanager
def change_dir(path):
    old_dir = os.getcwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(old_dir)

print("Before: ", os.getcwd())

with change_dir("/tmp"):
    print("Inside: ", os.getcwd())

print("After: ", os.getcwd())


Before:  /Users/sanjoypator/Desktop/dev/se/backend/backend/WebDevLearning/notes/phase-1-python-fundamentals/week-1/notebooks
Inside:  /private/tmp
After:  /Users/sanjoypator/Desktop/dev/se/backend/backend/WebDevLearning/notes/phase-1-python-fundamentals/week-1/notebooks


**What you’ll observe**

- Directory changes inside the block

- Automatically restored afterward

### Example 3: Temporary Environment Variables

This context manager sets an environment variable temporarily and restores
the original value after the block exits.

In [26]:
import os
from contextlib import contextmanager

@contextmanager
def temporary_env_var(key, value):
    old_value = os.environ.get(key)
    os.environ[key] = value
    try:
        yield
    finally:
        if old_value is None:
            del os.environ[key]
        else:
            os.environ[key] = old_value

print("Before: ",os.environ.get("DEBUG"))

with temporary_env_var("DEBUG", "true"):
    print("Inside: ", os.environ.get("DEBUG"))

print("After: ", os.environ.get("DEBUG"))

Before:  None
Inside:  true
After:  None


**What you’ll observe**

- Variable exists only inside the with block

- Original environment is restored

### Key Takeaways

- Context managers guarantee cleanup
- They prevent bugs caused by forgotten resets
- Ideal for temporary state changes and resource handling
- Cleaner and safer than manual `try/finally`
